### Block 0 — Install packages

In [22]:
!pip install -q openai datasets scikit-learn tqdm pandas


### Block 1 — Imports and DeepSeek API client

In [23]:
import os
import getpass

from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

from openai import OpenAI

# Read DeepSeek API key (not your OpenAI key)
os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Paste your DeepSeek API key here: ")

# Create DeepSeek client using OpenAI-compatible SDK
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

# DeepSeek chat model (V3 family)
MODEL_NAME = "deepseek-chat"


Paste your DeepSeek API key here: ··········


### Block 2 — Load FPB dataset

In [24]:
dataset = load_dataset("ChanceFocus/en-fpb", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3 — Labels and normalization

In [19]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map the raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Handle short variants like "pos" or "neg"
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4 — DeepSeek sentiment classifier

In [25]:
def build_prompt(text: str) -> str:
    """
    Build the instruction prompt for sentiment classification.
    """
    return (
        "You are an expert financial sentiment classifier.\n\n"
        "Classify the sentiment of the following financial news snippet as exactly one of:\n"
        "negative, neutral, positive.\n\n"
        "Return only one word: negative, neutral, or positive.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )


def predict_label(text: str) -> str:
    """
    Call DeepSeek (official API) and return a normalized label.
    """
    prompt = build_prompt(text)

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a financial sentiment classifier. "
                    "Always reply with exactly one word: negative, neutral, or positive."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_tokens=8,
        temperature=0.0,
    )

    raw = completion.choices[0].message.content
    return normalize_prediction(raw)


### Block 5 — Evaluation loop over FPB test split

In [27]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example["text"]
    true_label = example["answer"].strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 970/970 [19:02<00:00,  1.18s/it]


### Block 6 — Compute metrics and inspect predictions

In [28]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.8072

Classification report:
              precision    recall  f1-score   support

    negative     0.8175    0.8879    0.8512       116
     neutral     0.8863    0.7834    0.8316       577
    positive     0.6826    0.8231    0.7463       277

    accuracy                         0.8072       970
   macro avg     0.7955    0.8315    0.8097       970
weighted avg     0.8199    0.8072    0.8096       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from L